# Experiment results

Interactive counterpart to `uv run -m experiments.results`. Reuses the exact same
aggregation/plotting code from `experiments/results.py` (`build_summary_rows`,
`plot_efficiency_frontier`, `plot_latency_distribution`, `plot_per_field_accuracy`)
so this notebook can never drift from the CLI's output -- it just gives you a live
`DataFrame` to filter/sort and inline plots to look at without opening PNGs.

Re-run the "select runs" cell after a new `runner.py`/`evaluate.py` pass to pick up
fresh data.

In [ ]:
import sys
from pathlib import Path

# Jupyter's cwd is usually the notebook's own directory (experiments/), not the repo
# root, so `import experiments...` fails unless the repo root is on sys.path.
for _candidate in (Path.cwd(), Path.cwd().parent):
    if (_candidate / "experiments" / "results.py").exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        break

import pandas as pd
from IPython.display import Image, display

from experiments.config import DEFAULT_EXPERIMENT, GROUND_TRUTH_KEY, MODEL_REGISTRY
from experiments.results import (
    DEFAULT_EVALUATIONS_ROOT,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_RUNS_ROOT,
    _discover_latest_runs,
    _raw_wall_seconds,
    build_summary_rows,
    plot_efficiency_frontier,
    plot_latency_distribution,
    plot_per_field_accuracy,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Select runs

Defaults to the most recent run per `model_key` under `experiments/runs`, matching
the CLI's default. To pin specific runs instead (mirrors `--run` on the CLI), set
`selected_runs` explicitly, e.g.:

```python
from experiments.results import _resolve_explicit_runs
selected_runs = _resolve_explicit_runs(["gpt-oss-120b__20260715T101500Z", ...], DEFAULT_RUNS_ROOT)
```

In [ ]:
selected_runs = _discover_latest_runs(DEFAULT_RUNS_ROOT)
if not selected_runs:
    raise SystemExit(
        f"No runs found under {DEFAULT_RUNS_ROOT} -- run experiments/runner.py first."
    )

for model_key, run_dir in selected_runs.items():
    print(f"{model_key:>16}  ->  {run_dir}")

## Summary table

Same rows `results.py` writes to `summary_table.csv` -- one row per model, timing +
per-field similarity to `GROUND_TRUTH_KEY` (`gpt-oss-120b`).

In [ ]:
rows = build_summary_rows(selected_runs, DEFAULT_EVALUATIONS_ROOT)
df = pd.DataFrame(rows)
display(df)

## Plots

`results.py`'s plot functions render to `matplotlib`'s non-interactive `Agg` backend
and save straight to PNG (no `plt.show()`), so we call them exactly as the CLI does
and then display the saved PNG inline -- pixel-identical to what `--output-dir` would
produce, no separate plotting code to keep in sync.

In [ ]:
plots_dir = DEFAULT_OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Time vs. accuracy
path = plots_dir / "efficiency_frontier.png"
plot_efficiency_frontier(rows, path)
display(Image(filename=str(path)))

In [ ]:
# Latency distribution
path = plots_dir / "latency_distribution.png"
plot_latency_distribution(selected_runs, path)
display(Image(filename=str(path)))

In [ ]:
# Per-field accuracy
path = plots_dir / "per_field_accuracy.png"
plot_per_field_accuracy(rows, path)
display(Image(filename=str(path)))

## Interactive exploration

`df`, `rows`, and `selected_runs` are all live in this kernel -- filter/sort/plot
however's useful. A couple of starting points:

In [ ]:
# Candidates only, ranked by overall similarity to ground truth
candidates = df[df["model_key"] != GROUND_TRUTH_KEY].sort_values("overall_similarity_mean", ascending=False)
display(candidates)

In [ ]:
# Raw per-document wall_seconds for one model, as a pandas Series (e.g. for a
# histogram, custom percentile, or outlier lookup that the boxplot above hides)
model_key = next(iter(selected_runs))
pd.Series(_raw_wall_seconds(selected_runs[model_key]), name=f"{model_key}_wall_seconds").describe()